In [1]:
%pip install langchain_core langchain_community langchain-groq
%pip install transformers torch sentencepiece
%pip install langchain

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Defaulting to user installation because normal site-packages is not writeable
  Using cached sentencepiece-0.2.0.tar.gz (2.6 MB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'error'
Note: you may need to restart the kernel to use updated packages.


  error: subprocess-exited-with-error
  
  × Getting requirements to build wheel did not run successfully.
  │ exit code: 1
  ╰─> [48 lines of output]
      Traceback (most recent call last):
        File "c:\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 353, in <module>
          main()
          ~~~~^^
        File "c:\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 335, in main
          json_out['return_val'] = hook(**hook_input['kwargs'])
                                   ~~~~^^^^^^^^^^^^^^^^^^^^^^^^
        File "c:\Python313\Lib\site-packages\pip\_vendor\pyproject_hooks\_in_process\_in_process.py", line 118, in get_requires_for_build_wheel
          return hook(config_settings)
        File "C:\Users\elmir\AppData\Local\Temp\pip-build-env-mdm59kve\overlay\Lib\site-packages\setuptools\build_meta.py", line 334, in get_requires_for_build_wheel
          return self._get_build_requires(config_sett

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
# from time import sleeps
model_choice = 0
match model_choice:
    case 0:
        from langchain_groq import ChatGroq

        #We used a low temperature here for the model since it doesn't need to be creative, it needs to extract features and be somewhat deterministic.

        llm = ChatGroq(
            temperature=0,
            groq_api_key="api_key",
            model_name="llama3.1-8b-8192",
        )

        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a feature analyst, you must understand text and extract apps' features."),
            ("user", "{input}\n{formatting}"),
        ])

        output_parser = StrOutputParser()

        chain = prompt | llm | output_parser

    case 1:
        from langchain_groq import ChatGroq

        #We used a low temperature here for the model since it doesn't need to be creative, it needs to extract features and be somewhat deterministic.
        llm = ChatGroq(
            temperature=0,
            groq_api_key="api_key",
            model_name="mixtral-8x7b-32768",
        )

        prompt = ChatPromptTemplate.from_messages([
            ("system", "You are a feature analyst, you must understand text and extract apps' features."),
            ("user", "{input}\n{formatting}"),
        ])

        output_parser = StrOutputParser()

        chain = prompt | llm | output_parser

In [10]:
import pandas as pd

filtered_apps = pd.read_csv("data\\all_apps_cleaned_df.csv")
i = 0

In [ ]:
import time

# for i, row in filtered_apps.iterrows():
filtered_apps['uncleaned_LLM_features'] = ""
for i in range(i, len(filtered_apps.index)):
    description = filtered_apps['description'][i]
    print(i)
    time.sleep(2)
    match model_choice:
        case 0:
            formatting = """
                You are an mobile application reviewer and you have been tasked with reviewing an application, your role is summerize the features of the application based on its description. Here is what you need to do:
                        - A feature should be a short phrase (10 words or less) in lowercase that describes a functionality of the app.
                        - Do not include single words as features.
                        - The output should be formatted as a python list. ONLY output this list, nothing else.
                        - NEVER CREATE A FEATURE THAT IS NOT SUPPORTED BY THE DESCRIPTION!
                  Example :[/INST]
                    24/7 customer support, easy-to-use interface, secure payment processing, using AI agent to help customers
            """
            input = f"""
                ### APP DESCRIPTION
                {description}
                ### END OF APP DESCRIPTION
            """
            response = chain.invoke({"input": input, "formatting": formatting})
            print(f"response type is : {type(response)} and response: {response} ")
            filtered_apps['uncleaned_LLM_features'][i] = response


In [12]:
filtered_apps.to_csv("data\\llm_features_uncleaned.csv", index=False)

In [15]:
### Number of raw features
import pandas as pd
import ast
import re
import warnings

# Suppress all warnings
warnings.filterwarnings("ignore")

def parse_list(s):
    # Use regex to match quoted strings correctly (single or double quotes)
    # Escape internal single quotes (like in "seek doctor's advice")
    list_str=  s.split(',')
    res = []
    for elm in list_str: 
        cleaned_string = re.sub(r'[^a-zA-Z0-9\s,]', '', elm)
        res.append(cleaned_string)
    # Now use ast.literal_eval safely
    return res

df = pd.read_csv("data\\llm_features_uncleaned.csv")

all_features = []
for i in range(len(df.index)):
    feature = parse_list(df["uncleaned_LLM_features"][i])
    all_features.extend(feature)
    df["uncleaned_LLM_features"][i] = feature

print(f"Number of raw features: {len(all_features)}")
print(f"Number of raw features: {len(list(set(all_features)))}")

Number of raw features: 1342
Number of raw features: 1254


In [16]:
import pandas as pd
import warnings

# Suppress warnings
warnings.filterwarnings("ignore")

# Step 1: Randomly select 12 rows
sampled_df = df.sample(n=12, random_state=42)

# Step 2: Expand lists into rows
exploded_df = sampled_df.explode('uncleaned_LLM_features').reset_index(drop=True)

# Step 3: Save to CSV
exploded_df.to_csv('fine_tuning_df.csv', index=False)

print(exploded_df)


                                         appId                         title  \
0    com.soft_solutions.alldieasesandtreatment  All Diseases Treatments 2025   
1    com.soft_solutions.alldieasesandtreatment  All Diseases Treatments 2025   
2    com.soft_solutions.alldieasesandtreatment  All Diseases Treatments 2025   
3    com.soft_solutions.alldieasesandtreatment  All Diseases Treatments 2025   
4    com.soft_solutions.alldieasesandtreatment  All Diseases Treatments 2025   
..                                         ...                           ...   
141                   com.ecommunity.community      Community Health Network   
142                   com.ecommunity.community      Community Health Network   
143                   com.ecommunity.community      Community Health Network   
144                   com.ecommunity.community      Community Health Network   
145                   com.ecommunity.community      Community Health Network   

     score    genre  price  free  \
0  

In [ ]:
import pandas as pd

df = pd.read_csv("data\\fine_tuning_df.csv")
len(df)

146

In [7]:
len(df["appId"].unique()) # 12 unique apps

12

In [5]:
df["Correct_feature"].value_counts()

Correct_feature
True     137
False      9
Name: count, dtype: int64